# HiconaGraph

In [1]:
# Import packages
import hicona as hi
import pandas as pd
import polars as pl
import graph_tool.all as gt

# Data paths 
COOLER_PATH = "../data/IMR90_100kb_chr17_balanced.cool"
TABLE_PATH = "../data/IMR90_chr17_annotated_table"

## Creating a HiconaGraph

The `HiconaGraph` class constructor takes three inputs, an iterable of bins, an iterables of pixels and a default count for added edges. When initializing the object, a graph (`graph_tool.Graph`) is created following these steps:

- The entries of the bin table are added as vertexes. All default columns (`bin_id`, `chrom`, `start`, `end`), as well as all annotation columns, are added as vertex properties. Moreover, the property `node_id` is also added; this is needed since `graph_tool` requires the vertexes to be numbered from 0 to `n`, where `n` is the total number of vertexes.
- The entries of the pixel table are added as edges. The process is the same with the bins. The property `is_genomic`, initialized to `False` for all edges, is also added.
- A check is performed that all bins, e.i. vertices, which are genomically contiguous (e.i. `chr1:0-10000` and `chr1:10000-20000`) have an edge between them. If an edge is not present, it is added with a count value equal to the provided default value. Also, the value of the property `is_genomic` for those edges is set to `True`.


<div class="alert alert-info">

Note 

We call <b>genomic links</b> the edges added between genomically contiguous nodes/bins which did not have an edge between them already. Genomic links are needed for some applications, such as clustering, where disconnected components mught lead to issues. Adding these edges is also justified by the fact that it is highly unlikely for genomically contiguous bins to not contact each other at all; when that happens, it is more likely to be some bias (for instance in read mapping).
</div>

The created graph is then stored in the `graph` attribute of the `HiconaCooler` instance.

In [2]:
table = hi.PixelTable.load(TABLE_PATH)
graph = hi.HiconaGraph(
    bins = table.bins.get_dataframe("chr17:0-10000000"),
    pixels = table.get_chunks("chr17:0-10000000"),
    default_link = 1,
)

graph.graph

<Graph object, undirected, with 100 vertices and 4853 edges, 5 internal vertex properties, 3 internal edge properties, at 0x752538ca9250>

Alternatively, the `HiconaGraph` instance can be created directly from a `PixelTable` instance via the `get_graph` method.

In [3]:
graph = table.get_graph("chr17:0-10000000")  # Same result but no control on genomic link
graph.graph 

<Graph object, undirected, with 100 vertices and 4853 edges, 5 internal vertex properties, 3 internal edge properties, at 0x7525376a89d0>

It is worth noting that `HiconaGraph` does not have any other attribute aside from `graph`. This is to ensure that, when saving the `graph_tool.Graph` to any common graph format such as `.xml`, all the information present in the graph is retained.

<div class="alert alert-warning">

Warning

`HiconaGraph` objects are not backed by a storage on disk, but rather a copy of the data in RAM. This means that all the data in the graph is kept in memory.
</div>

## Reading data

To retrieve bin and pixel information from a `HiconaGraph` instance (supposedly after some form of manipulation), use the `get_bins` and `get_pixels` methods.

In [4]:
for bin_chunk in graph.get_bins():
    break 
bin_chunk.head()

bin_id,chrom,start,end,weight
i64,str,i32,i32,f64
26475,"""chr17""",0,100000,null
26476,"""chr17""",100000,200000,0.007338
26477,"""chr17""",200000,300000,0.006855
26478,"""chr17""",300000,400000,0.005692
26479,"""chr17""",400000,500000,0.007215


In [5]:
for pix_chunk in graph.get_pixels():
    break
pix_chunk.head()

bin1_id,bin2_id,count,rand_int
i64,i64,i32,i64
26475,26481,1,47
26475,26518,1,53
26475,26526,1,92
26475,26534,1,95
26476,26477,4003,2


Just like other data accessors provided by HiCONA, there are some flags which can be passed to customize data retrieval:

- `chunk_size`: maximum number of lines per fetched chunk
- `dtype`: data type of the individual chunks fetched (`polars` dataframe or `pandas` dataframe)
- `bare`: (only for bins) return only the base columns (`bin_id`, `chrom`, `start`, `end`) with no annotation
- `drop_node_id`: (only for bins) whether to drop the node id which was assigned
- `keep_genomic`: (only for pixels) whether to return the edges which were added as genomic links or discard them

These methods return generators to avoid having an additional full copy of the data into memory. If you graph is sufficiently small, you can wrap the generator with a concat method to obtain a single dataframe.

In [6]:
pl.concat(graph.get_pixels()).head()

bin1_id,bin2_id,count,rand_int
i64,i64,i32,i64
26475,26481,1,47
26475,26518,1,53
26475,26526,1,92
26475,26534,1,95
26476,26477,4003,2


In [7]:
pd.concat(graph.get_pixels(dtype="pandas")).head()

,bin1_id,bin2_id,count,rand_int
0,26475,26481,1,47
1,26475,26518,1,53
2,26475,26526,1,92
3,26475,26534,1,95
4,26476,26477,4003,2


## `graph_tool` API

As mentioned in the **Base Implementation** section of the documentation, `graph_tool.Graph` provides a very flexible and low-level interface. The downside of this interface is that it takes a while to get used to and can be quite overwhelming.

In [8]:
len(tuple(filter(lambda x: not x.startswith("_"), dir(graph.graph))))  # Number of public methods of graph_tool.Graph

86

As you can see, the `graph_tool.Graph` class has a lot of public methods just for basic graph manipulation. `HiconaGraph` wants to provide higher order functions to simplify the handling of Hi-C data derived networks; these mostly entail graph creation, data retrieval in bin/pixel compatible format, specialized versions of generic functions (e.i. clustering). Consider though the following scenarios:

- You really need a specialized version of an algorithm currently not present in HiCONA
- You need to apply an algorithm which is not chromatin data specific
- You simply want to explore your data

In these cases, you are probably better off just using the `graph_tool` API directly, by implementing your own functions acting directly on the `graph_tool.Graph` instance stored in the `graph` attribute of `HiconaGraph`.

As an example, assume you want to compute the [betweenness centrality](https://en.wikipedia.org/wiki/Betweenness_centrality) for each bin. Using the `graph_tool` API this can be achieved as follows:

In [9]:
betweenness_map, _ = gt.betweenness(graph.graph, weight=graph.graph.ep.count)
graph.graph.vp["betweenness"] = betweenness_map

The previous block of code computed and stored the betweenness in a new property map. This information can be then accessed using the usual HiCONA API.

In [10]:
pl.concat(graph.get_bins())

bin_id,chrom,start,end,weight,betweenness
i64,str,i32,i32,f64,f64
26475,"""chr17""",0,100000,null,0.115906
26476,"""chr17""",100000,200000,0.007338,0.116947
26477,"""chr17""",200000,300000,0.006855,0.004054
26478,"""chr17""",300000,400000,0.005692,0.000258
26479,"""chr17""",400000,500000,0.007215,0.069036
…,…,…,…,…,…
26570,"""chr17""",9500000,9600000,0.004251,0.018084
26571,"""chr17""",9600000,9700000,0.005525,0.181382
26572,"""chr17""",9700000,9800000,0.004759,0.093806
